# Weeks 13-22 environment check

Run this notebook on the course Databricks cluster as ANY student (or as the cluster login - falls through to `student-01`). Probes every AWS resource and credential path the Weeks 13-22 notebooks depend on. Each section prints PASS or raises with a clear message.

**What this notebook proves**:
1. Per-student AWS keys load from `aws-course-creds-NN`
2. Class-wide config loads from `aws-course-shared`
3. boto3 STS handshake works
4. Bedrock Haiku 3 + Sonnet 4.5 + Titan v2 + Cohere rerank invokable
5. Bedrock Knowledge Base (`knowledge-base-id` secret) reachable and Retrieve works
6. AgentCore Memory (`week16_fraud_investigation`) discoverable
7. SageMaker MLflow tracking server reachable
8. SageMaker fraud-classifier-endpoint InService and invokable
9. SageMaker model-package-group + Model Monitor schedule visible
10. CloudWatch FraudClassifier/Accuracy metric has a data point
11. SNS bread-academy-class-alerts publishable
12. MWAA bread-academy-airflow AVAILABLE
13. S3 bread-academy-shared and bread-academy-airflow-dags reachable
14. Unity Catalog `bread_academy.course_data.fraud_transactions` queryable

If a section fails, the error message tells you what to ask the instructor to fix (which secret, which AWS resource, which IAM grant).

In [ ]:
# Library pins consistent with W19/W20 working state. No apache-airflow
# (DAGs run on MWAA), no strands (this is an env probe, not an agent).
# sagemaker-mlflow plugin is required for mlflow client to talk to a
# SageMaker MLflow tracking server via its ARN.
%pip install --quiet \
    "numpy<2" "pandas<2" \
    "boto3>=1.36" \
    "sagemaker>=2.230,<3" \
    "mlflow-skinny>=2.13,<3" \
    "sagemaker-mlflow>=0.1.0" \
    "requests>=2.31"

dbutils.library.restartPython()

from importlib.metadata import version
for pkg in ["boto3", "sagemaker", "mlflow-skinny", "sagemaker-mlflow", "requests"]:
    try: print(f"{pkg:25s} {version(pkg)}")
    except Exception as e: print(f"{pkg:25s} NOT INSTALLED ({e})")


## 1 - Identity + per-student credential load

The Week 19-22 setup pattern: derive student number from Databricks user (or smoke override), then pull AWS keys from `aws-course-creds-NN` and class config from `aws-course-shared`.

In [ ]:
import os, json, time, base64, boto3
from botocore.exceptions import ClientError

_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
_num = os.environ.get("BREAD_SMOKE_STUDENT_ID") or (
    _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
)
creds_scope = f"aws-course-creds-{_num}"
print(f"User: {_user} -> creds_scope={creds_scope}")

try:
    AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
    AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
    AWS_REGION            = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")
except Exception as e:
    raise RuntimeError(
        f"FAIL secret load: {e}. Ask instructor to populate {creds_scope} "
        "with aws-access-key-id + aws-secret-access-key, and aws-course-shared "
        "with aws-region."
    )

os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"]            = AWS_REGION
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION
os.environ["AWS_REGION_NAME"]       = AWS_REGION  # litellm reads this one

sts = boto3.client("sts", region_name=AWS_REGION)
ident = sts.get_caller_identity()
assert ident["Account"] == "962804699607", f"wrong account: {ident['Account']}"
print(f"PASS 1.1 - STS caller: {ident['Arn']}")

## 2 - Class-wide secret bundle (`aws-course-shared`)

Every secret key the W13-W22 notebooks pull from `aws-course-shared`. Each is mandatory for at least one week.

In [ ]:
REQUIRED_SHARED_KEYS = [
    "aws-region",                       # W19-W22
    "sagemaker-execution-role-arn",     # W19-W22
    "sns-alerts-topic-arn",             # W20+W21
    "course-s3-bucket",                 # W20
    "shared-endpoint-name",             # W19+W20
    "knowledge-base-id",                # W13/W17/W18
    "langfuse-public-key",              # W20
    "langfuse-secret-key",              # W20
    "langfuse-host",                    # W20
    "mlflow-tracking-server-arn",       # W19
]
loaded = {}
missing = []
for k in REQUIRED_SHARED_KEYS:
    try:
        loaded[k] = dbutils.secrets.get(scope="aws-course-shared", key=k)
    except Exception:
        missing.append(k)

print(f"loaded {len(loaded)} / {len(REQUIRED_SHARED_KEYS)} keys")
if missing:
    print("MISSING:", missing)
    raise RuntimeError(
        "Ask instructor to run scripts/instructor_setup_aws.py to populate "
        "these aws-course-shared keys: " + ", ".join(missing)
    )
print("PASS 2 - all class-wide secrets present")

# Expose typed shortcuts
SAGEMAKER_ROLE_ARN = loaded["sagemaker-execution-role-arn"]
SNS_TOPIC_ARN      = loaded["sns-alerts-topic-arn"]
COURSE_S3_BUCKET   = loaded["course-s3-bucket"]
ENDPOINT_NAME      = loaded["shared-endpoint-name"]
KB_ID              = loaded["knowledge-base-id"]
MLFLOW_ARN         = loaded["mlflow-tracking-server-arn"]

## 3 - Bedrock invocation (Sonnet 4.5 primary, Titan v2, Cohere rerank)

The datacouch account has **Sonnet 4.5** model access enabled, not Haiku 3 (Marketplace subscription not in place for Haiku). Sonnet 4.5 is what every week's LLM call should use.

If Sonnet 4.5 fails with `AccessDeniedException`, the instructor needs to enable model access in the Bedrock console.

In [ ]:
br = boto3.client("bedrock-runtime", region_name=AWS_REGION)
br_agent = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)

# 3.1 Sonnet 4.5 - PRIMARY LLM for every week in this account
try:
    r = br.converse(
        modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        messages=[{"role":"user","content":[{"text":"ping"}]}],
        inferenceConfig={"maxTokens": 5, "temperature": 0},
    )
    print(f"PASS 3.1 - Sonnet 4.5: {r['output']['message']['content'][0]['text'][:50]!r}")
except ClientError as e:
    raise RuntimeError(f"Sonnet 4.5 invoke failed: {e}. Ask instructor to enable Bedrock model access for Sonnet 4.5.")

# 3.2 Haiku 3 - OPTIONAL (Marketplace subscription required; not enabled in datacouch)
try:
    r = br.converse(
        modelId="us.anthropic.claude-3-haiku-20240307-v1:0",
        messages=[{"role":"user","content":[{"text":"ping"}]}],
        inferenceConfig={"maxTokens": 5, "temperature": 0},
    )
    print(f"PASS 3.2 - Haiku 3: {r['output']['message']['content'][0]['text'][:50]!r}")
except ClientError as e:
    print(f"SKIP 3.2 - Haiku 3 access not granted ({e.response['Error']['Code']}) - Sonnet 4.5 is used instead in datacouch")

# 3.3 Titan embed v2 (Week 17/18 embeddings, also W22 deep-dive)
try:
    r = br.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": "smoke test"}),
    )
    body = json.loads(r["body"].read())
    print(f"PASS 3.3 - Titan v2 embed: dim={len(body['embedding'])}")
except ClientError as e:
    raise RuntimeError(f"Titan v2 embed failed: {e}")

# 3.4 Cohere rerank (Week 18)
try:
    r = br_agent.rerank(
        queries=[{"type":"TEXT","textQuery":{"text":"fraud policy"}}],
        sources=[
            {"type":"INLINE","inlineDocumentSource":{"type":"TEXT","textDocument":{"text":"high-amount transactions need step-up"}}},
            {"type":"INLINE","inlineDocumentSource":{"type":"TEXT","textDocument":{"text":"unrelated baking recipe"}}},
        ],
        rerankingConfiguration={
            "type":"BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration":{
                "numberOfResults":2,
                "modelConfiguration":{"modelArn":f"arn:aws:bedrock:{AWS_REGION}::foundation-model/cohere.rerank-v3-5:0"},
            }
        },
    )
    print(f"PASS 3.4 - Cohere rerank: top result index={r['results'][0]['index']}")
except ClientError as e:
    print(f"SKIP 3.4 - Cohere rerank failed ({e.response['Error']['Code']}) - only W18 uses it")

## 4 - Bedrock Knowledge Base + Retrieve (W13, W17, W18)

Confirms the KB is `ACTIVE` and a Retrieve query returns results from the ingested fraud-policy corpus.

In [ ]:
br_agent_mgmt = boto3.client("bedrock-agent", region_name=AWS_REGION)
try:
    kb = br_agent_mgmt.get_knowledge_base(knowledgeBaseId=KB_ID)
    status = kb["knowledgeBase"]["status"]
    print(f"KB {KB_ID}: status={status}")
    assert status == "ACTIVE", f"KB not ACTIVE: {status}"
except ClientError as e:
    raise RuntimeError(f"KB get failed: {e}. Run scripts/build_kb.py to (re)create.")

try:
    r = br_agent.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": "what is the fraud policy for high-amount transactions?"},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
    )
    hits = r.get("retrievalResults", [])
    if not hits:
        raise RuntimeError("Retrieve returned 0 hits - corpus not ingested? Re-run scripts/build_kb.py")
    print(f"PASS 4 - KB Retrieve returned {len(hits)} hits; top score={hits[0].get('score',0):.3f}")
except ClientError as e:
    raise RuntimeError(f"KB Retrieve failed: {e}")

## 5 - AgentCore Memory (W16)

Confirms the `week16_fraud_investigation` memory exists and is ACTIVE.

In [ ]:
ac = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)
try:
    mems = ac.list_memories().get("memories", [])
except ClientError as e:
    raise RuntimeError(f"list_memories failed: {e}. Ask instructor to run scripts/rebuild_baseline_infra.py.")

hit = next((m for m in mems if "week16_fraud_investigation" in (m.get("id") or m.get("name") or "")), None)
if not hit:
    raise RuntimeError(
        "AgentCore Memory 'week16_fraud_investigation' missing. "
        "Ask instructor to run scripts/rebuild_baseline_infra.py."
    )
print(f"PASS 5 - AgentCore Memory: id={hit.get('id')} status={hit.get('status')}")

## 6 - SageMaker MLflow tracking server (W19)

Confirms the server is Created/Started and we can connect.

In [ ]:
sm = boto3.client("sagemaker", region_name=AWS_REGION)
try:
    mlflow_srv = sm.describe_mlflow_tracking_server(
        TrackingServerName=MLFLOW_ARN.split("/")[-1]
    )
    print(f"PASS 6 - MLflow server: status={mlflow_srv['TrackingServerStatus']}")
    assert mlflow_srv["TrackingServerStatus"] in ("Created", "Started"), \
        f"server not ready: {mlflow_srv['TrackingServerStatus']}"
except ClientError as e:
    raise RuntimeError(f"MLflow server describe failed: {e}")

# Lightweight client probe via the plugin (sagemaker-mlflow SigV4-signs the
# request). search_experiments is a known-noisy probe on fresh servers - some
# servers 403 it from non-creator principals even when sagemaker-mlflow:* is
# granted. We tolerate that with SKIP because the actual W19 labs use
# mlflow.set_tracking_uri() + mlflow.start_run() (writes), not searches.
try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_ARN)
    client = mlflow.MlflowClient(MLFLOW_ARN)
    # get_experiment_by_name with a known-default name is the lightest call
    # the plugin supports. Returns None on missing, raises on auth failure.
    _ = client.get_experiment_by_name("Default")
    print("PASS 6.1 - MLflow client reachable and authorized")
except Exception as e:
    msg = str(e)[:200]
    if "403" in msg or "Access Denied" in msg or "Unauthorized" in msg:
        print(f"SKIP 6.1 - mlflow client authorized at IAM but server returned 403 - "
              "tolerable; W19 labs use mlflow.start_run() (writes) which DOES work "
              "for the principal that created the experiment")
    else:
        print(f"SKIP 6.1 - mlflow client probe failed ({msg}) - server is ready (Step 6 PASSED), only client wiring failed")

## 7 - SageMaker fraud-classifier-endpoint (W19+W20+W21+W22)

Endpoint must be `InService`. Then we invoke it on one synthetic narrative.

In [ ]:
smr = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
try:
    ep = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"endpoint {ENDPOINT_NAME}: {ep['EndpointStatus']}")
    assert ep["EndpointStatus"] == "InService", f"not InService: {ep['EndpointStatus']}"
except ClientError as e:
    raise RuntimeError(f"describe_endpoint failed: {e}. Ask instructor to run scripts/instructor_setup_aws.py.")

narrative = (
    "Card-not-present purchase of $4823 at an overseas crypto exchange "
    "by a customer whose account is 12 days old."
)
try:
    r = smr.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": narrative}),
    )
    out = json.loads(r["Body"].read())
    print(f"PASS 7 - endpoint invoke: {out}")
except ClientError as e:
    raise RuntimeError(f"endpoint invoke failed: {e}")

## 8 - SageMaker model package group + monitoring schedule (W19+W20)

In [ ]:
try:
    mpg = sm.describe_model_package_group(ModelPackageGroupName="fraud-classifier-week19")
    print(f"model package group: {mpg['ModelPackageGroupStatus']}")
except ClientError as e:
    raise RuntimeError(f"model package group missing: {e}")

try:
    mps = sm.list_model_packages(ModelPackageGroupName="fraud-classifier-week19").get("ModelPackageSummaryList", [])
    print(f"  versions: {len(mps)} (latest approval={mps[0]['ModelApprovalStatus'] if mps else 'none'})")
except ClientError as e:
    print(f"list_model_packages failed: {e}")

try:
    schedules = sm.list_monitoring_schedules().get("MonitoringScheduleSummaries", [])
    sch = next((s for s in schedules if s["MonitoringScheduleName"] == "fraud-classifier-hourly"), None)
    if sch:
        print(f"PASS 8 - monitoring schedule: {sch['MonitoringScheduleStatus']}")
    else:
        print("INFO 8 - no 'fraud-classifier-hourly' schedule (W20 lab creates it on first run)")
except ClientError as e:
    print(f"list_monitoring_schedules failed: {e}")

## 9 - CloudWatch + SNS (W20+W21)

Verifies the `FraudClassifier/Accuracy` metric namespace has data points (W21 Lab 3 gate depends on this) and that we can publish to the alerts topic (W20+W21).

In [ ]:
cw  = boto3.client("cloudwatch", region_name=AWS_REGION)
sns = boto3.client("sns", region_name=AWS_REGION)

try:
    mets = cw.list_metrics(Namespace="FraudClassifier", MetricName="Accuracy").get("Metrics", [])
    if mets:
        print(f"PASS 9.1 - FraudClassifier/Accuracy metric registered ({len(mets)} dimensions)")
    else:
        print("WARN 9.1 - no FraudClassifier/Accuracy data points - W21 Lab 3 gate needs a seed (instructor: rebuild_baseline_infra.py publishes one)")
except ClientError as e:
    print(f"list_metrics failed: {e}")

try:
    sns.publish(
        TopicArn=SNS_TOPIC_ARN,
        Subject="smoke test",
        Message=f"smoke test from student-{_num} at {time.time():.0f}",
    )
    print(f"PASS 9.2 - SNS publish ok to {SNS_TOPIC_ARN}")
except ClientError as e:
    raise RuntimeError(f"SNS publish failed: {e}")

## 10 - MWAA + S3 (W21+W22)

Confirms MWAA env is AVAILABLE, the DAGs bucket is reachable, and the per-student DAG-upload prefix is writable.

In [ ]:
mwaa = boto3.client("mwaa", region_name=AWS_REGION)
s3   = boto3.client("s3",   region_name=AWS_REGION)

try:
    env = mwaa.get_environment(Name="bread-academy-airflow")["Environment"]
    print(f"MWAA bread-academy-airflow: status={env['Status']} version={env.get('AirflowVersion')}")
    assert env["Status"] == "AVAILABLE", f"MWAA not AVAILABLE: {env['Status']}"
except ClientError as e:
    raise RuntimeError(f"MWAA get_environment failed: {e}")

for b in ("bread-academy-shared", "bread-academy-airflow-dags"):
    try:
        s3.head_bucket(Bucket=b)
        print(f"S3 {b}: reachable")
    except ClientError as e:
        raise RuntimeError(f"S3 {b}: {e}")

test_key = f"dags/student_{_num}/_smoke_probe_{int(time.time())}.txt"
try:
    s3.put_object(Bucket="bread-academy-airflow-dags", Key=test_key, Body=b"smoke")
    s3.delete_object(Bucket="bread-academy-airflow-dags", Key=test_key)
    print(f"PASS 10 - per-student DAG prefix writable: dags/student_{_num}/")
except ClientError as e:
    raise RuntimeError(f"per-student S3 PutObject failed: {e}")

## 11 - Unity Catalog + Spark read (W19+W20+W21+W22)

All Databricks-side weeks read from `bread_academy.course_data.fraud_transactions` via Spark. This section:
1. Verifies the table exists and the cluster can `spark.read.table(...)` it
2. Performs a real Spark transformation (groupBy aggregate) to confirm executors work
3. Pulls a small slice via `toPandas()` (the conversion path students use)
4. Validates the schema matches what the W19/W20 training notebooks expect

In [ ]:
from pyspark.sql import functions as F

# 11.1 - table exists, can be read
try:
    df = spark.read.table("bread_academy.course_data.fraud_transactions")
    n = df.count()
    cols = df.columns
    print(f"PASS 11.1 - fraud_transactions: {n:,} rows; {len(cols)} columns")
except Exception as e:
    raise RuntimeError(
        f"Unity Catalog read failed: {e}. Ask instructor to run "
        "scripts/instructor_setup_databricks.ipynb on the cluster."
    )

# 11.2 - schema matches what W19/W20 training notebooks expect
EXPECTED_COLUMNS = {
    "transaction_id", "customer_id", "amount", "merchant_country",
    "merchant_category", "hour_of_day", "is_weekend",
    "days_since_last_txn", "narrative", "is_fraud", "partition_date",
}
present = set(cols)
missing = EXPECTED_COLUMNS - present
if missing:
    raise RuntimeError(
        f"FAIL 11.2 - missing expected columns: {missing}. "
        "Ask instructor to re-run scripts/instructor_setup_databricks.ipynb."
    )
print(f"PASS 11.2 - all {len(EXPECTED_COLUMNS)} expected columns present")

# 11.3 - real Spark transformation (proves executors work, not just driver)
fraud_rate_by_country = (
    df.groupBy("merchant_country")
      .agg(
          F.count("*").alias("txn_count"),
          F.avg(F.col("is_fraud").cast("double")).alias("fraud_rate"),
      )
      .orderBy(F.col("fraud_rate").desc())
)
fraud_rate_by_country.show(5, truncate=False)
print(f"PASS 11.3 - Spark groupBy aggregate succeeded")

# 11.4 - toPandas() conversion path (this is what W19 training uses to
# build the train.csv that gets uploaded to S3 for the SageMaker training job)
sample_pdf = df.limit(200).toPandas()
assert len(sample_pdf) == 200, f"expected 200 rows, got {len(sample_pdf)}"
assert set(sample_pdf.columns) >= {"narrative", "is_fraud"}, "missing fields needed for fine-tune"
print(f"PASS 11.4 - toPandas() ok: sample shape={sample_pdf.shape}")

## 12 - Remote SageMaker training (Spark -> S3 -> training job)

The Week 19 critical path: pull data from Spark, write to S3, submit a SageMaker training job using `SageMakerStudentExecutionRole`.

**Instance**: `ml.g4dn.xlarge` (GPU, ~$0.526/hour). The AWS HuggingFace TRAINING DLC catalog ships **only GPU + Neuron containers** - there are no CPU training images, regardless of transformers/pytorch version. The W19 main notebook uses the same GPU instance.

**Cost**: tiny job (200 rows, 5 grad updates) finishes in ~8-10 min including container pull. ~$0.10 per run.

**Bypass**: set `BREAD_SMOKE_SKIP_TRAINING=1` in env to skip this section.

In [ ]:
if os.environ.get("BREAD_SMOKE_SKIP_TRAINING") == "1":
    print("SKIP 12 - BREAD_SMOKE_SKIP_TRAINING=1 set; not submitting remote training")
else:
    import sagemaker, tempfile
    from sagemaker.huggingface import HuggingFace

    # 12.1 - upload the 200-row Pandas sample to S3 as CSV
    smoke_prefix = f"smoke/student-{_num}/{int(time.time())}"
    train_uri = f"s3://bread-academy-shared/{smoke_prefix}/train.csv"

    train_pdf = sample_pdf[["narrative", "is_fraud"]].rename(
        columns={"narrative": "text", "is_fraud": "label"}
    )
    csv_bytes = train_pdf.to_csv(index=False).encode()
    s3.put_object(
        Bucket="bread-academy-shared",
        Key=f"{smoke_prefix}/train.csv",
        Body=csv_bytes,
    )
    print(f"PASS 12.1 - uploaded train.csv to {train_uri} ({len(csv_bytes):,} bytes)")

    # 12.2 - write source_dir to a local temp directory (NOT /dbfs - that's
    # read-only on Unity-Catalog-enabled workspaces).
    SOURCE_DIR = tempfile.mkdtemp(prefix="bread_academy_smoke_")
    train_script = '''
import argparse, json, os
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--model_name_or_path", default="distilbert-base-uncased")
    p.add_argument("--epochs", type=int, default=1)
    p.add_argument("--train_batch_size", type=int, default=16)
    p.add_argument("--learning_rate", type=float, default=5e-5)
    p.add_argument("--max_steps", type=int, default=5)
    args = p.parse_args()

    train_dir = os.environ["SM_CHANNEL_TRAIN"]
    out_dir   = os.environ["SM_MODEL_DIR"]
    df = pd.read_csv(os.path.join(train_dir, "train.csv"))
    ds = Dataset.from_pandas(df)
    tok = AutoTokenizer.from_pretrained(args.model_name_or_path)
    def _t(b): return tok(b["text"], padding="max_length", truncation=True, max_length=64)
    ds = ds.map(_t, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(args.model_name_or_path, num_labels=2)
    tr_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.train_batch_size,
        learning_rate=args.learning_rate,
        max_steps=args.max_steps,
        logging_steps=1,
        save_strategy="no",
        report_to=[],
    )
    Trainer(model=model, args=tr_args, train_dataset=ds).train()
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "smoke_marker.json"), "w") as f:
        json.dump({"smoke": True, "steps": args.max_steps}, f)
'''
    with open(os.path.join(SOURCE_DIR, "train.py"), "w") as f:
        f.write(train_script)
    with open(os.path.join(SOURCE_DIR, "requirements.txt"), "w") as f:
        f.write("datasets>=2.18\npandas>=1.5,<2\n")
    print(f"PASS 12.2 - source_dir prepared at {SOURCE_DIR}")

    # 12.3 - build the HuggingFace estimator.
    # GPU instance is REQUIRED - the AWS HuggingFace training DLC catalog has
    # zero CPU images. transformers 4.36.0 / pytorch 2.1.0 / py310 is the same
    # combo Week 19 uses for the real fraud-classifier training and is
    # documented to have a GPU container.
    sm_session = sagemaker.Session(boto_session=boto3.Session(region_name=AWS_REGION))
    job_name = f"smoke-fraud-{_num}-{int(time.time())}"
    estimator = HuggingFace(
        entry_point="train.py",
        source_dir=SOURCE_DIR,
        role=SAGEMAKER_ROLE_ARN,
        instance_type="ml.g4dn.xlarge",
        instance_count=1,
        transformers_version="4.36.0",
        pytorch_version="2.1.0",
        py_version="py310",
        hyperparameters={
            "model_name_or_path": "distilbert-base-uncased",
            "epochs": 1,
            "train_batch_size": 16,
            "learning_rate": 5e-5,
            "max_steps": 5,  # tiny smoke - 5 grad updates
        },
        sagemaker_session=sm_session,
        max_run=900,  # 15 min cap; GPU container pull is the slow part
        output_path=f"s3://bread-academy-shared/{smoke_prefix}/output/",
        base_job_name="smoke-fraud",
    )

    # 12.4 - submit + wait (~8-10 min: container pull dominates, training is seconds)
    print(f"PASS 12.4a - submitting training job {job_name} (this can take ~10 min)...")
    try:
        estimator.fit({"train": f"s3://bread-academy-shared/{smoke_prefix}/"}, job_name=job_name, wait=True)
        print(f"PASS 12.4b - training job {job_name} COMPLETED")
        print(f"  output: {estimator.model_data}")
    except Exception as e:
        raise RuntimeError(
            f"FAIL 12.4 - training failed: {e}. "
            f"Check CloudWatch logs at /aws/sagemaker/TrainingJobs/{job_name}."
        )
    finally:
        # Cleanup S3 smoke prefix
        try:
            for obj in s3.list_objects_v2(Bucket="bread-academy-shared", Prefix=smoke_prefix).get("Contents", []):
                s3.delete_object(Bucket="bread-academy-shared", Key=obj["Key"])
            print(f"  cleanup: deleted s3://bread-academy-shared/{smoke_prefix}/")
        except Exception as ce:
            print(f"  cleanup warning: {ce}")

## Done

If every section above printed PASS (or a tolerated SKIP/WARN), this Databricks workspace is ready to run the W13-W22 student notebooks end-to-end against the rebuilt AWS environment.

In [ ]:
print("=" * 60)
print("WEEKS 13-22 ENVIRONMENT CHECK COMPLETE")
print("=" * 60)
print(f"  student id      : {_num}")
print(f"  AWS account     : 962804699607 (datacouch)")
print(f"  AWS region      : {AWS_REGION}")
print(f"  KB id           : {KB_ID}")
print(f"  endpoint        : {ENDPOINT_NAME}")
print(f"  MWAA env        : bread-academy-airflow")
print(f"  MLflow server   : {MLFLOW_ARN.split('/')[-1]}")
print("  ready to run    : Weeks 13-22")